
# Computer Vision Using Deep Learning - Practical 7

# CNN for Image Classification with PyTorch

1.  **MNIST Dataset:** Building a basic CNN to classify handwritten digits and experimenting with various optimization algorithms.
2.  **Fashion-MNIST Dataset:** Applying the same architecture to a more challenging dataset of clothing items.
3.  **CIFAR-10 Dataset:** Modifying the CNN to handle colored, more complex images.

## 1. Setup and Imports

First, let's import all the necessary libraries. We'll need `torch` for building the network, `torchvision` for datasets and image transformations, and `matplotlib` for visualization.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
from torchsummary import summary

# Set device to GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


---

## 2. Image Classification on the MNIST Dataset

The MNIST dataset consists of 28x28 grayscale images of handwritten digits (0-9). It's a classic dataset for getting started with image classification.

### 2.1. Load Data and Preprocessing

In [ ]:
transform_mnist = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_data_mnist = datasets.MNIST(root='data', train=True, download=True, transform=transform_mnist)
train_loader_mnist = DataLoader(train_data_mnist, batch_size=64, shuffle=True)

test_data_mnist = datasets.MNIST(root='data', train=False, download=True, transform=transform_mnist)
test_loader_mnist = DataLoader(test_data_mnist, batch_size=64, shuffle=False)

### 2.2. CNN Model Architecture

Here's a simple CNN architecture:
1.  **Conv1**: A convolutional layer with 1 input channel (grayscale), 32 output channels, a 3x3 kernel.
2.  **ReLU**: A non-linear activation function.
3.  **MaxPool**: A 2x2 max pooling layer to downsample the feature map.
4.  **Conv2**: Another convolutional layer with 32 input channels, 64 output channels, a 3x3 kernel.
5.  **Flatten**: The output is flattened to a 1D vector.
6.  **FC1**: A fully connected layer with 128 neurons.
7.  **FC2**: The final output layer with 10 neurons, one for each digit class.

In [3]:

class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        # Input shape: (batch_size, 1, 28, 28)
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, stride=1, padding=1)
        # Shape after conv1: (batch_size, 32, 28, 28)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        # Shape after pool: (batch_size, 32, 14, 14)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1)
        # Shape after conv2: (batch_size, 64, 14, 14)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        # Shape after pool: (batch_size, 64, 7, 7)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)  # Flattened size is 64*7*7
        self.fc2 = nn.Linear(128, 10) # 10 output classes

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 64 * 7 * 7) # Flatten the tensor
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x
    
mnist_model = CNN().to(device)

print("--- MNIST / Fashion-MNIST CNN Summary ---")
# The input size is (channels, height, width)
summary(mnist_model, (1, 28, 28))

--- MNIST / Fashion-MNIST CNN Summary ---
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1           [-1, 32, 28, 28]             320
         MaxPool2d-2           [-1, 32, 14, 14]               0
            Conv2d-3           [-1, 64, 14, 14]          18,496
         MaxPool2d-4             [-1, 64, 7, 7]               0
            Linear-5                  [-1, 128]         401,536
            Linear-6                   [-1, 10]           1,290
Total params: 421,642
Trainable params: 421,642
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.36
Params size (MB): 1.61
Estimated Total Size (MB): 1.97
----------------------------------------------------------------


### 2.3. Training and Testing Functions

We'll create reusable functions for the training loop and the evaluation process.

In [4]:
def train(model, train_loader, optimizer, criterion, epoch):
    """Function to handle the training loop for one epoch."""
    model.train() 
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        
        # Forward pass
        optimizer.zero_grad() 
        output = model(data)
        loss = criterion(output, target)
        
        # Backward pass and optimization
        loss.backward()
        optimizer.step()

        if batch_idx % 200 == 0:
            print(f"Epoch {epoch} [{batch_idx*len(data)}/{len(train_loader.dataset)}]\tLoss: {loss.item():.4f}")

def test(model, test_loader, criterion):
    """Function to evaluate the model on the test set."""
    model.eval() # Set the model to evaluation mode
    test_loss = 0
    correct = 0

    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += criterion(output, target).item() * data.size(0)
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(test_loader.dataset)
    accuracy = 100. * correct / len(test_loader.dataset)
    print(f"\nTest set: Average loss: {test_loss:.4f}, Accuracy: {correct}/{len(test_loader.dataset)} ({accuracy:.2f}%)\n")

### 2.4. Experiment: Comparing Optimizers

#### Optimizer 1: Adam (Adaptive Moment Estimation) 

In [5]:
print("--- Training with Adam Optimizer ---")
model_adam = CNN().to(device)
optimizer = optim.Adam(model_adam.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

for epoch in range(1, 3): # Train for 2 epochs
    train(model_adam, train_loader_mnist, optimizer, criterion, epoch)
    test(model_adam, test_loader_mnist, criterion)

--- Training with Adam Optimizer ---
Epoch 1 [0/60000]	Loss: 2.2867
Epoch 1 [12800/60000]	Loss: 0.1457
Epoch 1 [25600/60000]	Loss: 0.1076
Epoch 1 [38400/60000]	Loss: 0.0102
Epoch 1 [51200/60000]	Loss: 0.0846

Test set: Average loss: 0.0414, Accuracy: 9870/10000 (98.70%)

Epoch 2 [0/60000]	Loss: 0.0270
Epoch 2 [12800/60000]	Loss: 0.0562
Epoch 2 [25600/60000]	Loss: 0.0067
Epoch 2 [38400/60000]	Loss: 0.0109
Epoch 2 [51200/60000]	Loss: 0.0067

Test set: Average loss: 0.0422, Accuracy: 9864/10000 (98.64%)



#### Optimizer 2: SGD with Momentum - Stochastic Gradient Descent (SGD) 

In [6]:
print("--- Training with SGD Optimizer ---")
model_sgd = CNN().to(device)
optimizer = optim.SGD(model_sgd.parameters(), lr=0.01, momentum=0.9)
criterion = nn.CrossEntropyLoss()

for epoch in range(1, 3):
    train(model_sgd, train_loader_mnist, optimizer, criterion, epoch)
    test(model_sgd, test_loader_mnist, criterion)

--- Training with SGD Optimizer ---
Epoch 1 [0/60000]	Loss: 2.2964
Epoch 1 [12800/60000]	Loss: 0.1403
Epoch 1 [25600/60000]	Loss: 0.1369
Epoch 1 [38400/60000]	Loss: 0.1361
Epoch 1 [51200/60000]	Loss: 0.0487

Test set: Average loss: 0.0500, Accuracy: 9841/10000 (98.41%)

Epoch 2 [0/60000]	Loss: 0.0306
Epoch 2 [12800/60000]	Loss: 0.1047
Epoch 2 [25600/60000]	Loss: 0.1646
Epoch 2 [38400/60000]	Loss: 0.0596
Epoch 2 [51200/60000]	Loss: 0.0070

Test set: Average loss: 0.0346, Accuracy: 9895/10000 (98.95%)



#### Optimizer 3: RMSprop

**RMSprop** (Root Mean Square Propagation) adapts the learning rate for each parameter, dividing it by a running average of the magnitudes of recent gradients.

In [7]:
print("--- Training with RMSprop Optimizer ---")
model_rmsprop = CNN().to(device)
optimizer = optim.RMSprop(model_rmsprop.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

for epoch in range(1, 3):
    train(model_rmsprop, train_loader_mnist, optimizer, criterion, epoch)
    test(model_rmsprop, test_loader_mnist, criterion)

--- Training with RMSprop Optimizer ---
Epoch 1 [0/60000]	Loss: 2.3176
Epoch 1 [12800/60000]	Loss: 0.0501
Epoch 1 [25600/60000]	Loss: 0.1015
Epoch 1 [38400/60000]	Loss: 0.1345
Epoch 1 [51200/60000]	Loss: 0.1403

Test set: Average loss: 0.0641, Accuracy: 9775/10000 (97.75%)

Epoch 2 [0/60000]	Loss: 0.0396
Epoch 2 [12800/60000]	Loss: 0.0440
Epoch 2 [25600/60000]	Loss: 0.0539
Epoch 2 [38400/60000]	Loss: 0.0430
Epoch 2 [51200/60000]	Loss: 0.0050

Test set: Average loss: 0.0279, Accuracy: 9903/10000 (99.03%)



#### Optimizer 4: Adagrad

**Adagrad** (Adaptive Gradient Algorithm) adapts the learning rate to the parameters, performing larger updates for infrequent and smaller updates for frequent parameters. It is well-suited for sparse data.

In [8]:
print("--- Training with Adagrad Optimizer ---")
model_adagrad = CNN().to(device)
optimizer = optim.Adagrad(model_adagrad.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

for epoch in range(1, 3):
    train(model_adagrad, train_loader_mnist, optimizer, criterion, epoch)
    test(model_adagrad, test_loader_mnist, criterion)

--- Training with Adagrad Optimizer ---
Epoch 1 [0/60000]	Loss: 2.2942
Epoch 1 [12800/60000]	Loss: 0.1278
Epoch 1 [25600/60000]	Loss: 0.1092
Epoch 1 [38400/60000]	Loss: 0.0801
Epoch 1 [51200/60000]	Loss: 0.1569

Test set: Average loss: 0.0462, Accuracy: 9845/10000 (98.45%)

Epoch 2 [0/60000]	Loss: 0.0237
Epoch 2 [12800/60000]	Loss: 0.0191
Epoch 2 [25600/60000]	Loss: 0.0812
Epoch 2 [38400/60000]	Loss: 0.0412
Epoch 2 [51200/60000]	Loss: 0.1475

Test set: Average loss: 0.0344, Accuracy: 9885/10000 (98.85%)



---

## 3. Applying the CNN to Fashion-MNIST

Fashion-MNIST is a drop-in replacement for MNIST but contains images of clothing items, making it a slightly more challenging task. We can use the **exact same model architecture** to see how it performs.

In [9]:
transform_fashion = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,)) # Mean and Std for Fashion-MNIST
])

train_data_fashion = datasets.FashionMNIST(root='data', train=True, download=True, transform=transform_fashion)
train_loader_fashion = DataLoader(train_data_fashion, batch_size=64, shuffle=True)

test_data_fashion = datasets.FashionMNIST(root='data', train=False, download=True, transform=transform_fashion)
test_loader_fashion = DataLoader(test_data_fashion, batch_size=64, shuffle=False)

In [10]:
print("--- Training on Fashion-MNIST with Adam ---")
model_fashion_adam = CNN().to(device)
optimizer = optim.Adam(model_fashion_adam.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

for epoch in range(1, 3):
    train(model_fashion_adam, train_loader_fashion, optimizer, criterion, epoch)
    test(model_fashion_adam, test_loader_fashion, criterion)

--- Training on Fashion-MNIST with Adam ---
Epoch 1 [0/60000]	Loss: 2.3045
Epoch 1 [12800/60000]	Loss: 0.3066
Epoch 1 [25600/60000]	Loss: 0.3204
Epoch 1 [38400/60000]	Loss: 0.3493
Epoch 1 [51200/60000]	Loss: 0.2224

Test set: Average loss: 0.3028, Accuracy: 8917/10000 (89.17%)

Epoch 2 [0/60000]	Loss: 0.2873
Epoch 2 [12800/60000]	Loss: 0.2446
Epoch 2 [25600/60000]	Loss: 0.4327
Epoch 2 [38400/60000]	Loss: 0.4164
Epoch 2 [51200/60000]	Loss: 0.2842

Test set: Average loss: 0.2768, Accuracy: 9018/10000 (90.18%)



In [11]:
print("--- Training on Fashion-MNIST with SGD ---")
model_fashion_sgd = CNN().to(device)
optimizer = optim.SGD(model_fashion_sgd.parameters(), lr=0.01, momentum=0.9)
criterion = nn.CrossEntropyLoss()

for epoch in range(1, 3):
    train(model_fashion_sgd, train_loader_fashion, optimizer, criterion, epoch)
    test(model_fashion_sgd, test_loader_fashion, criterion)

--- Training on Fashion-MNIST with SGD ---
Epoch 1 [0/60000]	Loss: 2.3108
Epoch 1 [12800/60000]	Loss: 0.5740
Epoch 1 [25600/60000]	Loss: 0.4498
Epoch 1 [38400/60000]	Loss: 0.5659
Epoch 1 [51200/60000]	Loss: 0.2484

Test set: Average loss: 0.3516, Accuracy: 8704/10000 (87.04%)

Epoch 2 [0/60000]	Loss: 0.2477
Epoch 2 [12800/60000]	Loss: 0.1634
Epoch 2 [25600/60000]	Loss: 0.3224
Epoch 2 [38400/60000]	Loss: 0.1752
Epoch 2 [51200/60000]	Loss: 0.3453

Test set: Average loss: 0.2915, Accuracy: 8936/10000 (89.36%)



### Fashion-MNIST Results:
You'll likely notice a drop in accuracy compared to MNIST (e.g., around 90%). This is expected as the dataset is more complex. This highlights the importance of tailoring model architecture and hyperparameters to the specific dataset.

---

## 4. Tackling a More Complex Dataset: CIFAR-10

The CIFAR-10 dataset contains 32x32 pixel **color** images across 10 classes (e.g., 'airplane', 'automobile', 'bird'). This requires a few modifications to our model:

1.  The first convolutional layer must accept **3 input channels** (for R, G, B) instead of 1.
2.  The input to the first fully connected layer will change because the image size is 32x32, not 28x28.

### 4.1. Data Loading and Preprocessing

In [12]:
# Normalization values for CIFAR-10 (3 channels)
transform_cifar = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261))
])

train_data_cifar = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_cifar)
train_loader_cifar = DataLoader(train_data_cifar, batch_size=64, shuffle=True)

test_data_cifar = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_cifar)
test_loader_cifar = DataLoader(test_data_cifar, batch_size=64, shuffle=False)

Files already downloaded and verified
Files already downloaded and verified


### 4.2. Modified CNN for CIFAR-10

Let's calculate the input size for the fully connected layer:
- Initial size: `32x32`
- After first pool: `16x16`
- After second pool: `8x8`

So, the flattened size will be `64 (channels) * 8 * 8`.

In [13]:
class CIFAR_CNN(nn.Module):
    def __init__(self):
        super(CIFAR_CNN, self).__init__()
        # Input shape: (batch_size, 3, 32, 32)
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1)
        # Shape after conv1: (batch_size, 32, 32, 32)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        # Shape after pool: (batch_size, 32, 16, 16)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        # Shape after conv2: (batch_size, 64, 16, 16)
        # Shape after pool: (batch_size, 64, 8, 8)
        self.fc1 = nn.Linear(64 * 8 * 8, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 64 * 8 * 8) # Flatten
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x
    
cifar_model = CIFAR_CNN().to(device)


print("\n--- CIFAR-10 CNN Summary ---")
# The input size is (3 channels, 32x32 pixels)
summary(cifar_model, (3, 32, 32))


--- CIFAR-10 CNN Summary ---
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1           [-1, 32, 32, 32]             896
         MaxPool2d-2           [-1, 32, 16, 16]               0
            Conv2d-3           [-1, 64, 16, 16]          18,496
         MaxPool2d-4             [-1, 64, 8, 8]               0
            Linear-5                  [-1, 128]         524,416
            Linear-6                   [-1, 10]           1,290
Total params: 545,098
Trainable params: 545,098
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.01
Forward/backward pass size (MB): 0.47
Params size (MB): 2.08
Estimated Total Size (MB): 2.56
----------------------------------------------------------------


### 4.3. Training and Evaluating the CIFAR-10 Model
CIFAR-10 is a more complex dataset, so we'll train for more epochs to achieve reasonable performance.

In [14]:
cifar_model = CIFAR_CNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(cifar_model.parameters(), lr=0.001)
epochs = 10

print("--- Training on CIFAR-10 ---")
for epoch in range(epochs):
    cifar_model.train()
    running_loss = 0.0
    for images, labels in train_loader_cifar:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = cifar_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(train_loader_cifar):.4f}")

print("\n--- Evaluating on CIFAR-10 Test Set ---")
cifar_model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader_cifar:
        images, labels = images.to(device), labels.to(device)
        outputs = cifar_model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Test Accuracy: {100 * correct / total:.2f}%")

# Save the trained model's state dictionary
torch.save(cifar_model.state_dict(), "cifar_cnn.pth")
print("\nModel saved to cifar_cnn.pth")

--- Training on CIFAR-10 ---
Epoch 1/10, Loss: 1.2894
Epoch 2/10, Loss: 0.9176
Epoch 3/10, Loss: 0.7637
Epoch 4/10, Loss: 0.6383
Epoch 5/10, Loss: 0.5326
Epoch 6/10, Loss: 0.4330
Epoch 7/10, Loss: 0.3454
Epoch 8/10, Loss: 0.2691
Epoch 9/10, Loss: 0.2064
Epoch 10/10, Loss: 0.1568

--- Evaluating on CIFAR-10 Test Set ---
Test Accuracy: 70.28%

Model saved to cifar_cnn.pth
